# 02 · Cortical parcellation for 19-channel EEG (source resolution; no labels)

1. **Source resolution per atlas** (68 Desikan-Killiany labels, 34 gyral groups, 18 regions, 12 lobes) for both
   inverses, from the fsaverage forward model with the stores' BEM and the stores' exact inverse settings: spatial
   rank of the 19-electrode leadfield, ROI-operator rank (forced redundancy), cross-talk, own-region dominance,
   leakage correlation. No EEG and no diagnosis is used.
2. **Atlas recommendation** by the pre-specified rule of `configs/final/selection.yaml` (`atlas_resolution`).
3. **Feature reliability** of the existing 68-ROI stores (development participants only, label-blind): split-half
   reliability of PSD / DE, node strength and edges; the rank of each participant's saved LCMV ROI operator.
   3b: whether coarser-atlas ROI signals could be recomputed from the stores' saved operators and ROI signals.

Building stores of a coarser atlas needs the author's approval (section 4 does nothing unless enabled). CPU runtime.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

## 1-2. Source resolution and the atlas recommendation

In [ ]:
sh("python scripts/final.py parcellation")

In [ ]:
import glob
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
import json
d = latest(FINAL + '/parcellation/resolution_*')
display(pd.read_csv(d + '/resolution_by_atlas.csv').round(3))
rec = json.load(open(d + '/recommendation.json'))
print('recommended atlas:', rec['recommended'], '|', rec['forward_model'])
for v in rec['verdicts']:
    print(v['atlas'], 'eligible' if v['eligible'] else 'not eligible', '; '.join(v['reasons']))

## 3. Feature reliability of the 68-ROI stores (development participants)

In [ ]:
for inverse in ('lcmv', 'wmne'):
    sh(f"python scripts/final.py reliability --inverse {inverse} --mode B")

## 3b. Can coarser-atlas ROI signals be rebuilt from the saved intermediates? (read-only)

Per development participant: rank of the saved ROI operator W, its response to a constant (average reference), the
relative residual of recovering the average-referenced sensor signals from the saved ROI signals (pinv(W)), and the
fraction of windows whose analytic-filter margins lie inside the saved pieces (only those could be recomputed
exactly without the recordings). Nothing is built; this only quantifies the shortcut (docs/final/parcellation.md §6).

In [ ]:
for inverse in ('lcmv', 'wmne'):
    sh(f"python scripts/final.py reconstruct-check --inverse {inverse} --mode B")

## 4. Stores of a coarser atlas (ONLY after the author's approval)

Builds the recommended atlas's stores for both inverses with the unchanged pipeline (merged Desikan-Killiany labels,
`mean_flip` over their own sources), validates them and matches wMNE to LCMV. New store directories; the 68-ROI
stores are untouched. CPU, about 40 min (LCMV) + 30-45 min (wMNE) per atlas. Then set `release.PRIMARY['atlas']`
by a logged decision (docs/final/parcellation.md).

In [ ]:
BUILD_COARSE_STORES = False      # set True only after the author approved the coarser atlas
ATLAS = rec['recommended']
if BUILD_COARSE_STORES and ATLAS and ATLAS != 'dk68':
    for inverse in ('lcmv', 'wmne'):
        sh(f"python scripts/build_stores.py build --inverse {inverse} --atlas {ATLAS}")
        sh(f"python scripts/build_stores.py validate --inverse {inverse} --atlas {ATLAS}")
    sh(f"python scripts/build_stores.py match --atlas {ATLAS}")
else:
    print('no store built (BUILD_COARSE_STORES is False or dk68 recommended)')